# Character Aggregator 테스트 (Production Level v1.1)

모든 7개 서브 에이전트 결과를 FullCharacter 형식으로 통합하는 Aggregator 테스트

## 역할: "Data Merger" (데이터 통합자)
- Identity, Appearance, Personality, Relations, Dialogue/Mood, Stats, Inventory 통합
- Safe Defaults 적용 (null → 기본값)
- 정규화된 필드명 사용

## v1.1.0 변경사항 검증:
- **Root-level 검색 필드**: `_id`, `name`, `role`, `level`, `faction`
- **Final Stats**: 아이템 보너스 반영된 최종 스탯
- **data_version**: 1.1.0

> **⚠️ 핵심 검증**: 모든 에이전트 데이터가 FullCharacter로 올바르게 병합되는지

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
# Mock data from each sub-agent (v1.1 - with item stats for bonus calculation)
MOCK_IDENTITY = {
    "아린": {"name": "아린", "age": 22, "gender": "female", "role": "protagonist", "aliases": [], "backstory": "은빛 여명 기사단 소속", "faction": "은빛 여명 기사단"},
    "카엘": {"name": "카엘", "age": 28, "gender": "male", "role": "antagonist", "aliases": ["배신자"], "backstory": "전직 기사, 암흑회 가담", "faction": "암흑회"}
}

MOCK_APPEARANCE = {
    "아린": {"hair_color": "검은색", "eyes": "갈색", "physique": "날씬한", "attire": ["은색 갑옷"]},
    "카엘": {"hair_color": "은색", "eyes": "빨간색", "physique": "건장한", "attire": ["검은 갑옷"]}
}

MOCK_PERSONALITY = {
    "아린": {"core_traits": ["용감함", "정의로운"], "flaws": ["고집"], "values": ["정의"]},
    "카엘": {"core_traits": ["냉정함", "야망있는"], "flaws": ["배신"], "values": ["권력"]}
}

MOCK_RELATIONS = {
    "아린": {"relations": [{"target": "카엘", "type": "ENEMY", "strength": 8}]},
    "카엘": {"relations": [{"target": "아린", "type": "ENEMY", "strength": 7}]}
}

MOCK_DIALOGUE_MOOD = {
    "아린": {"current_mood": {"emotion": "분노", "intensity": 7}, "dialogue": {"tone": "단호한", "catchphrases": []}},
    "카엘": {"current_mood": {"emotion": "냉소", "intensity": 5}, "dialogue": {"tone": "차가운", "catchphrases": []}}
}

MOCK_STATS = {
    "아린": {"stats": {"level": 25, "strength": 15}, "state": {"hp": 450, "hp_max": 500}, "combat": {"base_attack": 100, "total_attack": 150, "base_defense": 80}},
    "카엘": {"stats": {"level": 30, "strength": 20}, "state": {"hp": 600, "hp_max": 600}, "combat": {"base_attack": 120, "total_attack": 200, "base_defense": 100}, "economy": {"gold": 500}}
}

# v1.1: Inventory with item stats for bonus calculation
MOCK_INVENTORY = {
    "아린": {
        "equipped_items": [
            {"name": "은빛 검", "item_type": "WEAPON", "slot": "MAIN_HAND", "stats": {"attack_bonus": 30, "defense_bonus": 0}},
            {"name": "기사단 갑옷", "item_type": "ARMOR", "slot": "CHEST", "stats": {"defense_bonus": 25, "hp_bonus": 50}}
        ],
        "bag_items": [{"name": "체력 물약", "item_type": "CONSUMABLE", "quantity": 5}],
        "quest_items": [{"name": "봉인된 편지", "item_type": "QUEST"}]
    },
    "카엘": {
        "equipped_items": [
            {"name": "어둠의 대검", "item_type": "WEAPON", "slot": "MAIN_HAND", "stats": {"attack_bonus": 50, "defense_bonus": 0}},
            {"name": "암흑 갑옷", "item_type": "ARMOR", "slot": "CHEST", "stats": {"defense_bonus": 40, "hp_bonus": 100}}
        ],
        "bag_items": [],
        "quest_items": []
    }
}

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Aggregator 실행

In [3]:
from app.agents.extraction.character.aggregator import character_aggregator_node

# Create mock state with all sub-agent results
mock_state = {
    "content": "테스트 스토리",
    "char_identity": MOCK_IDENTITY,
    "char_appearance": MOCK_APPEARANCE,
    "char_personality": MOCK_PERSONALITY,
    "char_relations": MOCK_RELATIONS,
    "char_dialogue_mood": MOCK_DIALOGUE_MOOD,
    "char_stats": MOCK_STATS,
    "char_inventory": MOCK_INVENTORY,
    "completed_agents": ["identity", "appearance", "personality", "relations", "dialogue_mood", "stats", "inventory"],
    "errors": [],
    "messages": []
}

async def test_aggregator():
    print("🔗 Aggregator 테스트...")
    return await character_aggregator_node(mock_state)

result = run_async(test_aggregator())

if result.get('errors'):
    print(f"\n❌ 에러 발생:")
    for err in result.get('errors', []):
        print(f"   {err}")
else:
    characters = result.get('extracted_characters', [])
    print(f"\n✅ 통합 완료:")
    print(f"   - 캐릭터 수: {len(characters)}개")
    print(f"   - 이름: {[c.get('profile', {}).get('name') for c in characters]}")

🔗 Aggregator 테스트...

✅ 통합 완료:
   - 캐릭터 수: 2개
   - 이름: ['카엘', '아린']


## 2. v1.1 신규 필드 검증 (Root-level 검색 필드)

In [4]:
print("="*70)
print("🆕 v1.1 Root-level 검색 필드 검증")
print("="*70)

characters = result.get('extracted_characters', [])

for char in characters:
    name = char.get('name')  # Root-level name
    print(f"\n🧑 {name}")
    
    # Check root-level fields (v1.1)
    root_fields = {
        "_id": char.get('_id'),
        "name": char.get('name'),
        "role": char.get('role'),
        "level": char.get('level'),
        "faction": char.get('faction')
    }
    
    print(f"   Root-level fields:")
    for field, value in root_fields.items():
        status = "✅" if value is not None else "⚠️ (None)"
        print(f"      {field}: {value} {status}")

🆕 v1.1 Root-level 검색 필드 검증

🧑 카엘
   Root-level fields:
      _id: char-카엘-001 ✅
      name: 카엘 ✅
      role: antagonist ✅
      level: 30 ✅
      faction: 암흑회 ✅

🧑 아린
   Root-level fields:
      _id: char-아린-002 ✅
      name: 아린 ✅
      role: protagonist ✅
      level: 25 ✅
      faction: 은빛 여명 기사단 ✅


## 3. v1.1 Final Stats 검증 (아이템 보너스 계산)

In [5]:
print("="*70)
print("🎯 v1.1 Final Stats 검증 (아이템 보너스 적용)")
print("="*70)

characters = result.get('extracted_characters', [])

for char in characters:
    name = char.get('name')
    print(f"\n🧑 {name}")
    
    final_stats = char.get('final_stats', {})
    details = final_stats.get('details', {})
    
    if final_stats:
        print(f"   Final Attack: {final_stats.get('attack')}")
        print(f"   Final Defense: {final_stats.get('defense')}")
        print(f"   Final HP Max: {final_stats.get('hp_max')}")
        print(f"")
        print(f"   Details:")
        print(f"      Base Attack: {details.get('base_attack')} + Item Bonus: {details.get('item_attack_bonus')}")
        print(f"      Base Defense: {details.get('base_defense')} + Item Bonus: {details.get('item_defense_bonus')}")
        print(f"      Base HP Max: {details.get('base_hp_max')} + Item Bonus: {details.get('item_hp_bonus')}")
    else:
        print(f"   ❌ final_stats not found!")

🎯 v1.1 Final Stats 검증 (아이템 보너스 적용)

🧑 카엘
   Final Attack: 250
   Final Defense: 50
   Final HP Max: 700

   Details:
      Base Attack: 120 + Item Bonus: 50
      Base Defense: 100 + Item Bonus: 40
      Base HP Max: 600 + Item Bonus: 100

🧑 아린
   Final Attack: 180
   Final Defense: 35
   Final HP Max: 550

   Details:
      Base Attack: 100 + Item Bonus: 30
      Base Defense: 80 + Item Bonus: 25
      Base HP Max: 500 + Item Bonus: 50


## 4. FullCharacter 구조 확인

In [6]:
characters = result.get('extracted_characters', [])

print("="*70)
print("📊 FullCharacter 구조 확인")
print("="*70)

for char in characters:
    profile = char.get('profile', {})
    print(f"\n🧑 {profile.get('name')}")
    print(f"   ID: {profile.get('character_id')}")
    print(f"   Role: {char.get('role')}")
    print(f"   Age/Gender: {profile.get('age')} / {profile.get('gender')}")
    print(f"   Faction: {char.get('faction')}")
    
    # Stats
    stats = char.get('stats', {})
    state = char.get('state', {})
    print(f"   Level: {stats.get('level')} | STR: {stats.get('strength')}")
    print(f"   HP: {state.get('hp')}/{state.get('hp_max')}")
    
    # Inventory
    inv = char.get('inventory', {})
    equipped = inv.get('equipped_items', [])
    if equipped:
        print(f"   Equipped: {[i.get('name') for i in equipped]}")

📊 FullCharacter 구조 확인

🧑 카엘
   ID: char-카엘-001
   Role: antagonist
   Age/Gender: 28 / male
   Faction: 암흑회
   Level: 30 | STR: 20
   HP: 600/600
   Equipped: ['어둠의 대검', '암흑 갑옷']

🧑 아린
   ID: char-아린-002
   Role: protagonist
   Age/Gender: 22 / female
   Faction: 은빛 여명 기사단
   Level: 25 | STR: 15
   HP: 450/500
   Equipped: ['은빛 검', '기사단 갑옷']


## 5. Safe Defaults 검증

In [7]:
print("="*70)
print("🛡️ Safe Defaults 검증")
print("="*70)

characters = result.get('extracted_characters', [])

for char in characters:
    name = char.get('profile', {}).get('name')
    stats = char.get('stats', {})
    combat = char.get('combat', {})
    
    print(f"\n🧑 {name}")
    
    # Check for null values that should have defaults
    issues = []
    if stats.get('dexterity') is None:
        issues.append("dexterity is None (should have default)")
    if combat.get('base_attack') is None and combat.get('total_attack') is None:
        issues.append("No attack value")
    
    if issues:
        for issue in issues:
            print(f"   ⚠️ {issue}")
    else:
        print(f"   ✅ Safe defaults applied correctly")

🛡️ Safe Defaults 검증

🧑 카엘
   ✅ Safe defaults applied correctly

🧑 아린
   ✅ Safe defaults applied correctly


## 6. Full JSON 출력 (첫 번째 캐릭터)

In [8]:
print("="*70)
print("📄 Full JSON Output (첫 번째 캐릭터)")
print("="*70)

characters = result.get('extracted_characters', [])
if characters:
    print(json.dumps(characters[0], ensure_ascii=False, indent=2))
else:
    print("{}")

📄 Full JSON Output (첫 번째 캐릭터)
{
  "_id": "char-카엘-001",
  "name": "카엘",
  "role": "antagonist",
  "level": 30,
  "faction": "암흑회",
  "profile": {
    "character_id": "char-카엘-001",
    "name": "카엘",
    "age": 28,
    "gender": "male",
    "race": null,
    "faction": "암흑회",
    "mbti": null,
    "personality": [
      "냉정함",
      "야망있는"
    ],
    "chapter_appearance": null,
    "backstory": "전직 기사, 암흑회 가담"
  },
  "aliases": [
    "배신자"
  ],
  "status": "alive",
  "appearance": {
    "physique": "건장한",
    "skin_tone": "unspecified",
    "eyes": "빨간색",
    "nose": "unspecified",
    "mouth": "unspecified",
    "hair_style": "unspecified",
    "hair_color": "은색",
    "attire": [
      "검은 갑옷"
    ],
    "expression": "neutral",
    "scars_tattoos": [],
    "cyberware": [],
    "hair_color_normalized": {
      "description": "unspecified",
      "hex_code": null,
      "category": "UNSPECIFIED"
    },
    "eye_color_normalized": {
      "description": "unspecified",
      "hex_code": n

## 7. Production 체크리스트 (v1.1)

In [9]:
print("="*70)
print("✅ Production 체크리스트 (v1.1)")
print("="*70)

characters = result.get('extracted_characters', [])
checks = []

# 1. 캐릭터 수
if len(characters) == 2:
    checks.append(("✅", "2 characters merged correctly"))
else:
    checks.append(("❌", f"Expected 2 characters, got {len(characters)}"))

if characters:
    first_char = characters[0]
    
    # 2. Profile 필드
    has_profile = all(c.get('profile', {}).get('character_id') for c in characters)
    if has_profile:
        checks.append(("✅", "All characters have character_id"))
    else:
        checks.append(("❌", "Missing character_id"))
    
    # 3. v1.1: Root-level 검색 필드
    has_root_id = all(c.get('_id') for c in characters)
    has_root_name = all(c.get('name') for c in characters)
    has_root_level = all(c.get('level') is not None for c in characters)
    if has_root_id and has_root_name and has_root_level:
        checks.append(("✅", "v1.1 Root-level search fields (_id, name, level) present"))
    else:
        checks.append(("❌", "v1.1 Root-level search fields missing"))
    
    # 4. v1.1: Final Stats
    has_final_stats = all(c.get('final_stats', {}).get('attack') is not None for c in characters)
    if has_final_stats:
        checks.append(("✅", "v1.1 final_stats with item bonuses calculated"))
    else:
        checks.append(("❌", "v1.1 final_stats missing"))
    
    # 5. Relations graph
    has_relations = all(c.get('relations', {}).get('graph') for c in characters)
    if has_relations:
        checks.append(("✅", "Relations.graph populated"))
    else:
        checks.append(("⚠️", "Relations.graph missing"))
    
    # 6. Stats (normalized names)
    first_stats = first_char.get('stats', {})
    if 'strength' in first_stats or 'level' in first_stats:
        checks.append(("✅", "Stats using normalized field names"))
    else:
        checks.append(("⚠️", "Stats field names not normalized"))
    
    # 7. Inventory structure
    first_inv = first_char.get('inventory', {})
    if 'equipped_items' in first_inv:
        checks.append(("✅", "Inventory structure correct"))
    else:
        checks.append(("❌", "Inventory structure incorrect"))
    
    # 8. Combat base/total separation
    first_combat = first_char.get('combat', {})
    if 'total_attack' in first_combat or 'base_attack' in first_combat:
        checks.append(("✅", "Combat using base/total separation"))
    else:
        checks.append(("⚠️", "Combat not using base/total separation"))
    
    # 9. v1.1: data_version
    data_version = first_char.get('meta', {}).get('data_version')
    if data_version == "1.1.0":
        checks.append(("✅", f"data_version is {data_version}"))
    else:
        checks.append(("❌", f"data_version should be 1.1.0, got {data_version}"))

print()
for status, msg in checks:
    print(f"{status} {msg}")

print("\n" + "=" * 70)
passed = sum(1 for s, _ in checks if s == "✅")
print(f"결과: {passed}/{len(checks)} checks passed")

✅ Production 체크리스트 (v1.1)

✅ 2 characters merged correctly
✅ All characters have character_id
✅ v1.1 Root-level search fields (_id, name, level) present
✅ v1.1 final_stats with item bonuses calculated
✅ Relations.graph populated
✅ Stats using normalized field names
✅ Inventory structure correct
✅ Combat using base/total separation
✅ data_version is 1.1.0

결과: 9/9 checks passed


## 8. 디버그 정보

In [10]:
print("="*70)
print("🔍 디버그 정보")
print("="*70)
print(f"Result keys: {result.keys()}")
print(f"Errors: {result.get('errors', [])}")
print(f"Completed agents: {result.get('completed_agents', [])}")
print(f"Messages: {result.get('messages', [])}")

🔍 디버그 정보
Result keys: dict_keys(['extracted_characters', 'completed_agents', 'messages'])
Errors: []
Completed agents: ['identity', 'appearance', 'personality', 'relations', 'dialogue_mood', 'stats', 'inventory', 'aggregator']
Messages: [{'role': 'aggregator', 'content': 'Merged 2 characters from 7 sub-agents'}]
